# Evaluating similar games

How good are the "similar games" on Checkpointd? This notebook measures it with a hand-written golden set: 30 games from different genres and eras, each with 5 games a fan would expect to see. The golden set was written before looking at any result.

Retrieval works like production: cosine distance over `voyage-4-lite` embeddings (512 dimensions), leaving out the game itself and games from the same franchise, because the game page already has a franchise section.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from resolve import ROOT, load_golden

K = 5
games = pd.read_parquet(ROOT / 'data' / 'raw' / 'games.parquet').reset_index(drop=True)
golden = load_golden(games)
position = pd.Series(games.index, index=games['game_id'])
len(games), golden['query'].nunique(), len(golden)

(31749, 30, 150)

## Retrieval and metrics

- **precision@5**: how many of the 5 results are in the golden set. Each game has exactly 5 expected games, so the maximum is 1.0.
- **hit@5**: the share of games with at least one expected game in the top 5.

A good result that isn't in the golden set counts as wrong, so these numbers are a lower bound on the real quality.

In [2]:
def normalize(matrix):
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


def allowed_mask(query_index):
    mask = np.ones(len(games), dtype=bool)
    mask[query_index] = False
    franchise = games.at[query_index, 'franchise']
    if pd.notna(franchise):
        mask &= (games['franchise'] != franchise).to_numpy()
    return mask


def top_k_by_score(scores, query_index):
    ranked = np.where(allowed_mask(query_index), scores, -np.inf)
    return np.argsort(-ranked, kind='stable')[:K]


def evaluate(retrieve):
    rows = []
    for query_id, expected in golden.groupby('query_id', sort=False):
        query_index = position[query_id]
        top = games.loc[retrieve(query_index), 'game_id'].tolist()
        hits = len(set(top) & set(expected['expected_id']))
        rows.append({
            'query': games.at[query_index, 'name'],
            'hits': hits,
            'precision_at_5': hits / K,
            'results': games.set_index('game_id').loc[top, 'name'].tolist(),
        })
    return pd.DataFrame(rows)

## Three versions

- **Production**: name, genres, themes and summary.
- **Without themes**: the same text without the themes line. Games that have no themes keep their embedding, because their text is identical.
- **Baseline**: no embeddings at all, just Jaccard overlap of genres and themes, with ties broken by popularity. If embeddings don't beat this, they aren't worth paying for.


In [3]:
production = normalize(np.array(games['embedding'].tolist(), dtype=np.float32))

replacements = pd.read_parquet(ROOT / 'data' / 'raw' / 'embeddings_without_themes.parquet')
without_themes = np.array(games['embedding'].tolist(), dtype=np.float32)
without_themes[position[replacements['game_id']].to_numpy()] = np.array(replacements['embedding'].tolist(), dtype=np.float32)
without_themes = normalize(without_themes)

tags = [set(g) | set(t) for g, t in zip(games['genres'], games['themes'])]
popularity = games['rating_count'].to_numpy(dtype=float)
tie_break = popularity / (popularity.max() + 1) * 1e-6


def jaccard_scores(query_index):
    own = tags[query_index]
    return np.array([len(own & t) / len(own | t) if own | t else 0.0 for t in tags]) + tie_break


runs = {
    'production': evaluate(lambda i: top_k_by_score(production @ production[i], i)),
    'without_themes': evaluate(lambda i: top_k_by_score(without_themes @ without_themes[i], i)),
    'baseline_jaccard': evaluate(lambda i: top_k_by_score(jaccard_scores(i), i)),
}
summary = pd.DataFrame({
    name: {
        'precision@5': run['precision_at_5'].mean(),
        'hit@5': (run['hits'] > 0).mean(),
        'games with 0 hits': int((run['hits'] == 0).sum()),
    }
    for name, run in runs.items()
}).T
summary

,precision@5,hit@5,games with 0 hits
production,0.126667,0.433333,17.0
without_themes,0.120000,0.400000,18.0
baseline_jaccard,0.046667,0.200000,24.0


## Is the difference real?

With only 30 games the average moves a lot by chance, so this resamples the games 10,000 times to get a 95% interval for each version and for the difference against production.

In [4]:
generator = np.random.default_rng(42)
samples = generator.integers(0, 30, size=(10_000, 30))
base = runs['production']['precision_at_5'].to_numpy()
rows = []
for name, run in runs.items():
    values = run['precision_at_5'].to_numpy()
    means = values[samples].mean(axis=1)
    differences = (values - base)[samples].mean(axis=1)
    rows.append({
        'version': name,
        'ci_low': np.percentile(means, 2.5),
        'ci_high': np.percentile(means, 97.5),
        'diff_vs_production': (values - base).mean(),
        'diff_low': np.percentile(differences, 2.5),
        'diff_high': np.percentile(differences, 97.5),
    })
pd.DataFrame(rows).round(3)

,version,ci_low,ci_high,diff_vs_production,diff_low,diff_high
0,production,0.073,0.187,0.000,0.000,0.000
1,without_themes,0.067,0.180,-0.007,-0.040,0.027
2,baseline_jaccard,0.013,0.087,-0.080,-0.147,-0.013


## Game by game

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 250)
per_game = runs['production'][['query', 'hits']].rename(columns={'hits': 'production'})
per_game['without_themes'] = runs['without_themes']['hits']
per_game['baseline_jaccard'] = runs['baseline_jaccard']['hits']
per_game.sort_values('production', ascending=False).reset_index(drop=True)

,query,production,without_themes,baseline_jaccard
0,The Witcher 3: Wild Hunt,2,2,0
1,Resident Evil,2,2,1
2,Overcooked!,2,1,0
3,Counter-Strike: Global Offensive,2,2,1
4,Gran Turismo 7,2,1,0
5,RollerCoaster Tycoon,2,2,0
6,Apollo Justice: Ace Attorney,1,2,0
7,FIFA 23,1,1,0
8,XCOM: Enemy Unknown,1,1,0
9,Guitar Hero III: Legends of Rock,1,2,1


## Worst cases in production

In [6]:
runs['production'].sort_values('hits').head(10)[['query', 'hits', 'results']].reset_index(drop=True)

,query,hits,results
0,Hollow Knight,0,"[Aeterna Noctis, Vernal Edge, Lone Fungus, Flynn: Son of Crimson, Doomblade]"
1,Dark Souls,0,"[Lords of the Fallen, Demon's Souls, Elden Ring, Demon's Souls, Shadow Tower: Abyss]"
2,Stardew Valley,0,"[Fields of Mistria, Harvest Moon, Story of Seasons: A Wonderful Life, Harvest Moon: One World, Harvest Moon: Skytree Village]"
3,Portal,0,"[Teleportal 2, Portal 64, Portal Stories: VR, Portal, Polarity]"
4,StarCraft II: Wings of Liberty,0,"[Conquest: Frontier Wars, Star Ruler 2, Arena Wars 2, Tempest Rising, Supreme Commander]"
5,Sid Meier's Civilization VI,0,"[Sid Meier's Civilization IV: Colonization, Civilization: Call to Power, Sid Meier's Civilization, Europa Universalis, Age of Civilizations]"
6,Doom,0,"[Doomed, Quantum Error, Alien: Rogue Incursion, C.O.R.E., Afterfall: Insanity]"
7,Super Mario Bros.,0,"[Super Mario Star Scramble 3, Super Princess Peach, Super Paper Mario, Super Mario Bros: Odyssey - Chapter 1, Mario Starcatcher 2]"
8,Hades,0,"[Black Myth: Heaven, Death Must Die, Children of Morta, Halls of Torment, Dungeon Souls]"
9,The Last of Us,0,"[Just Survive, The Evil Within, Resident Evil: Revelations, ZombiU, Survival Zombies: The Inverted Evolution]"


## What was tried to improve it

These come from `scripts/run_experiments.py`, `scripts/finetune.py` and `scripts/quality_floor.py`. Settings were chosen on a development split built from IGDB's similar games lists, then checked once on a separate test split and on the golden set.

In [7]:
import json

fine_tuning = json.loads((ROOT / 'data' / 'finetune_report.json').read_text())
floor = json.loads((ROOT / 'data' / 'quality_floor_report.json').read_text())
rows = [
    {'version': name, 'golden': v['golden'][0], 'igdb_test': v['igdb_test'][0], 'golden_hit_rate': v['golden_hit_rate']}
    for name, v in fine_tuning['versions'].items()
]
rows += [
    {'version': f"voyage + floor of {floor[name]['floor']} ratings", 'golden': floor[name]['golden'][0], 'igdb_test': floor[name]['igdb_test'][0], 'golden_hit_rate': floor[name]['golden_hit_rate']}
    for name in ['floor_1', 'selected']
]
pd.DataFrame(rows).round(3)

,version,golden,igdb_test,golden_hit_rate
0,voyage_production,0.127,0.036,0.433
1,bge_base,0.053,0.015,0.233
2,bge_finetuned,0.020,0.074,0.100
3,voyage + floor of 1 ratings,0.133,0.039,0.467
4,voyage + floor of 10 ratings,0.180,0.041,0.567


The fine-tuned model is the main lesson here. Trained on IGDB's lists, it got better at reproducing those lists and worse on the human golden set. Those lists are a noisy proxy, so it learned the proxy instead of the goal. It wasn't chosen, and the rating floor was chosen instead.